# Exercise 21.1: A complex cardiac calcium model

In this exercise, you will wire together the **Jafri-Winslow-Borg (1998)** cardiomyocyte model. This model differentiates between 4 spatial calcium compartments ($\mathrm{Ca_{SS}}$, $\mathrm{Ca_{i}}$, $\mathrm{Ca_{JSR}}$, $\mathrm{Ca_{NSR}}$) and contains 31 state variables.

Because implementing 31 ODEs by hand guarantees typos, the core logic has been isolated into an external file (`Jafri_model.py`). Your job is to fill in the missing complex formulations: the RyR Markov states, the Luo-Rudy NCX, the compartmental calcium ODEs, and the transmembrane potential.

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from Jafri_model import Jafri_model_parts

## Exercise 21.1a: The ryanodine receptors

The RyR model used by Jafri et al. is based on the Keizer and Levine model. This model consists of 4 states (2 open and 2 closed states). The transitions between $P_{C1}$ and $P_{O1}$ and from $P_{O1}$ to $P_{O2}$ are calcium dependent. Since the RyRs are placed on the subspace the calcium dependency is based on the subspace calcium concentration $\mathrm{Ca}^{2+}_{\mathrm{SS}}$.

![RyR 4-state Markov model](../../fig/RyR_Jafri.png)

In order to go from closed to the first open calcium state, an increase in subspace calcium is needed. In the first open state $P_{O1}$ the receptor can close by adapting to its transition state $P_{C2}$. However, if the $\mathrm{Ca}^{2+}_{\mathrm{SS}}$ further increases, the channel reopens to state $P_{O2}$. This mechanism allows the modelling of calcium-induced calcium release.

1. Write out the ODEs for the four state variables on paper.
2. Based on the model diagram, decide which calcium concentration should be used to determine the opening of the ryanodine receptors.
3. Implement the RyR model below by filling in the `...` placeholders.

The release flux is: $J_{\mathrm{rel}} = v_1 \cdot (P_{O1} + P_{O2}) \cdot ([\mathrm{Ca}^{2+}]_{\mathrm{JSR}} - [\mathrm{Ca}^{2+}]_{\mathrm{SS}})$

## Exercise 21.1b: The NCX formulation

In Chapter 20, we covered the Luo-Rudy (1994) NCX formulation:

$$
I_{\mathrm{NaCa}} = k_{\mathrm{NaCa}}
\frac{
[\mathrm{Na}^+]^3_{\mathrm{i}} [\mathrm{Ca}^{2+}]_{\mathrm{e}} \cdot \exp\left[\eta V \frac{F}{RT}\right]
- [\mathrm{Na}^+]^3_{\mathrm{e}} [\mathrm{Ca}^{2+}]_{\mathrm{i}} \cdot \exp\left[(\eta-1) V \frac{F}{RT}\right]
}{
\left(K_{\mathrm{m, Na}}^3 + [\mathrm{Na}^+]_{\mathrm{e}}^3\right)
\left(K_{\mathrm{m, Ca}} + [\mathrm{Ca}^{2+}]_{\mathrm{e}}\right)
\left(1 + k_{\mathrm{sat}} \cdot \exp\left[(\eta-1) V \frac{F}{RT}\right]\right)}
$$

Implement this formulation in the code below by filling in the `...` placeholders.

## Exercise 21.1c: The calcium subsystem

This model differentiates between 4 calcium compartments. $\mathrm{Ca}^{2+}_{\mathrm{SS}}$ and $\mathrm{Ca}^{2+}_{\mathrm{i}}$ represent the subspace and bulk intracellular calcium concentrations respectively. $\mathrm{Ca}^{2+}_{\mathrm{JSR}}$ and $\mathrm{Ca}^{2+}_{\mathrm{NSR}}$ represent the calcium in the JSR (junctional SR) and NSR (network SR).

Based on the following diagram, write the equations for the 4 different calcium concentration changes, as we did in Exercise 21.1b.

![Jafri extended compartment model](../../fig/Jafri_extended.png)

Note that the transmembrane currents are given in Amperes (charge per second). However, we need concentration changes in molar (mol/L). Use the unit conversion factors `conv_Amp_SS` and `conv_Amp_myo` (which depend on the subspace and myoplasm volumes respectively).

Keep in mind the different compartmental volumes, and note that buffers ($B_i$, $B_{\mathrm{SS}}$, $B_{\mathrm{JSR}}$) have been pre-filled. Fill in which currents and fluxes affect each compartment.

## Exercise 21.1d: Calculating the transmembrane potential

Recall from Module L03 that the transmembrane potential can be calculated with the total membrane current and the membrane capacitance:

$$
\frac{\mathrm{d}V}{\mathrm{d}t} = \frac{I_{\mathrm{stim}} - I_{\mathrm{m}}}{C_{\mathrm{m}}} 
$$

Based on this equation and the compartment diagram above, fill in the transmembrane currents into the code.

## Complete RHS function

Now fill in the `...` placeholders below to complete the full right-hand-side function.

In [ ]:
def rhs_jafri(t, y):
    # Split up the 31-state vector
    (
        V, Nai, m, h, j, O, O_Ca,
        C0, C1, C2, C3, C4,
        C_Ca0, C_Ca1, C_Ca2, C_Ca3, C_Ca4,
        Ca_SS, Ko, Ki, y_gate, X, Cai,
        P_O1, P_O2, P_C1, P_C2,
        Ca_JSR, Ca_NSR, HTRPNCa, LTRPNCa,
    ) = y

    # Constants
    R, T, F, Cm = 8.3145e3, 310, 9.6845e4, 0.01
    Nao, Cao = 140, 1.8
    k_NaCa, K_mNa, K_mCa, k_sat, eta = 50, 87.5, 1.38, 0.1, 0.35
    v1, v2, v3 = 1.8, 0.58e-4, 1.8e-3
    k_a_plus, k_a_minus = 1.215e10, 0.1425
    k_b_plus, k_b_minus = 4.05e7, 1.93
    k_c_plus, k_c_minus = 0.018, 0.0008
    nCa, mCa = 4, 3
    tau_tr, tau_xfer = 34.48, 3.125
    K_mup, K_mCMDN, K_mCSQN = 0.5e-3, 2.38e-3, 0.8
    CSQN_tot, CMDN_tot = 15, 0.05
    Am, V_myo = 546.69, 0.92
    V_SS = 5.828e-05 * V_myo
    V_NSR = 0.081 * V_myo
    V_JSR = 0.00464 * V_myo
    conv_Amp_myo = Am / (2.0 * V_myo * F)
    conv_Amp_SS = Am / (2.0 * V_SS * F)

    # Stimulus (periodic pacing)
    stim_start, stim_end = 100, 600
    stim_period, stim_duration = 120, 1.0
    stim_amplitude = 0.516289
    if t >= stim_start and t <= stim_end and (t - stim_start) - math.floor((t - stim_start) / stim_period) * stim_period <= stim_duration:
        I_stim = stim_amplitude
    else:
        I_stim = 0

    # 1. Fetch pre-computed currents from the external file
    (
        dm_dt, dh_dt, dj_dt, i_Na,
        dX_dt, i_K, i_K1, i_Kp,
        i_NaK, i_ns_Ca, i_ns_Na, i_ns_K,
        i_p_Ca, i_Ca_b, i_Na_b,
        dy_dt,
        dC0_dt, dC1_dt, dC2_dt, dC3_dt, dC4_dt,
        dC_Ca0_dt, dC_Ca1_dt, dC_Ca2_dt, dC_Ca3_dt, dC_Ca4_dt,
        dO_dt, dO_Ca_dt,
        dHTRPNCa_dt, dLTRPNCa_dt,
        i_Ca_L_Ca, i_Ca_L_K, J_trpn,
    ) = Jafri_model_parts().currents_concentrations(
        V, m, h, j, Nai, X, Ko, Ki, Cai, y_gate,
        C0, C1, C2, C3, C4,
        C_Ca0, C_Ca1, C_Ca2, C_Ca3, C_Ca4,
        O, O_Ca, Ca_SS, Ca_JSR, Ca_NSR, HTRPNCa, LTRPNCa,
    )

    # ──────────────────────────────────────────────
    # Exercise 21.1a: RyR 4-state Markov model
    # ──────────────────────────────────────────────
    RyR_open = ...  # total open probability
    J_rel = ...     # release flux

    dP_C1_dt = ...
    dP_O1_dt = ...
    dP_O2_dt = ...
    dP_C2_dt = ...

    # ──────────────────────────────────────────────
    # Exercise 21.1b: NCX Luo-Rudy formulation
    # ──────────────────────────────────────────────
    i_NaCa = ...

    # ──────────────────────────────────────────────
    # Exercise 21.1c: Calcium subsystem
    # ──────────────────────────────────────────────
    J_leak = v2 * (Ca_NSR - Cai)
    J_up = (v3 * (Cai**2.0)) / ((K_mup**2.0) + (Cai**2.0))
    J_tr = (Ca_NSR - Ca_JSR) / tau_tr
    J_xfer = (Ca_SS - Cai) / tau_xfer

    # Buffer factors (pre-filled)
    Bi = 1.0 / (1.0 + (CMDN_tot * K_mCMDN) / ((K_mCMDN + Cai) ** 2.0))
    B_JSR = 1.0 / (1.0 + (CSQN_tot * K_mCSQN) / ((K_mCSQN + Ca_JSR) ** 2.0))
    B_SS = 1.0 / (1.0 + (CMDN_tot * K_mCMDN) / ((K_mCMDN + Ca_SS) ** 2.0))

    # Fill in the calcium compartment ODEs
    dCa_SS_dt = B_SS * (...)
    dCa_JSR_dt = B_JSR * (...)
    dCa_NSR_dt = ...
    dCai_dt = Bi * (...)

    # ──────────────────────────────────────────────
    # Exercise 21.1d: Transmembrane potential
    # ──────────────────────────────────────────────
    dV_dt = (I_stim - (...)) / Cm

    # Na and K concentration dynamics
    dNai_dt = -(i_Na + i_Na_b + i_ns_Na + i_NaCa * 3.0 + i_NaK * 3.0) * 2 * conv_Amp_myo
    dKi_dt = -(i_Ca_L_K + i_K + i_K1 + i_Kp + i_ns_K - i_NaK * 2.0) * 2 * conv_Amp_myo
    dKo_dt = (i_Ca_L_K + i_K + i_K1 + i_Kp + i_ns_K - i_NaK * 2.0) * 2 * conv_Amp_myo

    # Return all 31 derivatives
    return [
        dV_dt, dNai_dt, dm_dt, dh_dt, dj_dt, dO_dt, dO_Ca_dt,
        dC0_dt, dC1_dt, dC2_dt, dC3_dt, dC4_dt,
        dC_Ca0_dt, dC_Ca1_dt, dC_Ca2_dt, dC_Ca3_dt, dC_Ca4_dt,
        dCa_SS_dt, dKo_dt, dKi_dt, dy_dt, dX_dt, dCai_dt,
        dP_O1_dt, dP_O2_dt, dP_C1_dt, dP_C2_dt,
        dCa_JSR_dt, dCa_NSR_dt, dHTRPNCa_dt, dLTRPNCa_dt,
    ]

## Solve and plot

Once you have completed the RHS function, run the cell below to solve the model and plot the results.

In [ ]:
# Initial conditions (31 state variables)
y0 = (
    -84.1638, 10.2042, 0.0328302, 0.988354, 0.99254,
    9.84546e-21, 0,
    0.997208, 6.38897e-5, 1.535e-9, 1.63909e-14, 6.56337e-20,
    2.72826e-3, 6.99215e-7, 6.71989e-11, 2.87031e-15, 4.59752e-20,
    1.36058e-4, 5.4, 143.727, 0.998983, 0.000928836,
    9.94893e-11,
    1.19168e-3, 6.30613e-9, 0.762527, 0.236283,
    1.17504, 1.243891, 0.13598, 0.00635,
)

T = (0, 600)
solution = solve_ivp(rhs_jafri, T, y0, max_step=1.0)

# Unpack for plotting
time = solution.t
V = solution.y[0]
Ca_SS = solution.y[17]
Cai = solution.y[22]

fig, axs = plt.subplots(3, 1, figsize=(10, 8), sharex=True)

axs[0].plot(time, V)
axs[0].set_ylabel("Transmembrane Voltage (mV)")
axs[0].set_xlim(0, 600)

axs[1].plot(time, Ca_SS)
axs[1].set_ylabel("$[Ca^{2+}]_{SS}$ (mM)")

axs[2].plot(time, Cai)
axs[2].set_ylabel("$[Ca^{2+}]_{i}$ (mM)")
axs[2].set_xlabel("Time (ms)")

plt.tight_layout()
plt.show()